<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L18_LoRA_Adapters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L18: LoRA Adapters - Parameter-Efficient Fine-Tuning

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 18 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Understand Low-Rank Adaptation (LoRA) and why it reduces parameters
- Use the PEFT library to apply LoRA to a HuggingFace model
- Compare LoRA vs full fine-tuning (memory, speed, quality)
- Choose rank and target modules for your use case

---

## Concept: Why LoRA?

**LoRA** injects small trainable matrices (low-rank) into attention layers so we only train a fraction of parameters. Benefits: less GPU memory, faster training, often similar quality to full fine-tuning. **PEFT** (HuggingFace) makes it easy to apply.

---

In [ ]:
!pip install transformers torch accelerate peft datasets -q
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: LoRA Config and Apply to Model

We wrap the base model with PEFT LoRA: specify rank, alpha, and which modules to adapt (usually query/value in attention).

---

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Part 2: Train LoRA on Classification

Train with the same Trainer API; only LoRA parameters are updated.

---

In [ ]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = load_dataset("imdb", split="train", trust_remote_code=True).select(range(500))
eval_ds = load_dataset("imdb", split="test", trust_remote_code=True).select(range(100))
train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
train_ds.set_format("torch")
eval_ds.set_format("torch")

args = TrainingArguments(output_dir="./out_lora_l18", num_train_epochs=2, per_device_train_batch_size=8, evaluation_strategy="epoch", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds)
trainer.train()
print(trainer.evaluate())

## Part 3: Save and Load LoRA Weights

Save only the LoRA adapter (tiny file); load base model + adapter for inference.

---

In [ ]:
model.save_pretrained("./lora_adapter_l18")
print("Adapter saved. Load with: model = PeftModel.from_pretrained(base_model, './lora_adapter_l18')")

## Exercises

1. **Compare LoRA vs full fine-tuning**: Same data and epochs; report trainable params, memory, and eval accuracy.
2. **Vary rank**: Train with r=4, 8, 16; compare accuracy and adapter size.
3. **Target more modules**: Add key and output projections; measure impact.

---

## Key Takeaways

1. LoRA trains low-rank matrices in selected layers (e.g. q, v); most weights stay frozen.
2. PEFT integrates with Trainer; use LoraConfig for rank, alpha, target_modules.
3. Save/load only adapters for deployment; merge into base model optionally.

---

## Next Lesson

**L19: BERT Family Models** – BERT, RoBERTa, ALBERT, DistilBERT for classification, NER, and QA.

---